# Meeting Transcript Summarizer

## 1 — Project Overview

This notebook builds a **meeting transcript summarizer** that converts raw meeting text into structured output:

- **Executive summary** (2-3 sentences)
- **Key decisions** with rationale
- **Action items** with owner, deadline, priority
- **Blockers** and risks
- **Open questions** that need follow-up

**Engineering patterns taught:**

| Pattern | Description |
|---------|-------------|
| Prompt decomposition | Breaking one big task into focused sub-prompts |
| Structured output (JSON) | Forcing LLM output into typed schemas |
| Baseline comparison | Naive rule-based vs. LLM-powered extraction |
| Evaluation rubric | Scoring extraction quality against ground truth |
| Graceful fallback | Deterministic extraction when LLM is unavailable |

**Stack:** Python standard library only. Ollama (optional) for LLM-powered extraction.

**Prerequisite (optional):** [Ollama](https://ollama.com/) installed with `ollama pull qwen3:8b`. The notebook runs fully without it using rule-based fallback.

## 2 — Learning Goals

By the end of this notebook you will be able to:

1. **Parse meeting transcripts** — extract speaker turns, timestamps, and dialogue structure
2. **Design decomposed prompts** — break summarization into sub-tasks (decisions, actions, blockers) instead of one mega-prompt
3. **Build structured output schemas** — define exactly what the LLM returns as JSON
4. **Compare naive vs. decomposed approaches** — understand why prompt decomposition improves extraction quality
5. **Evaluate extraction quality** — score completeness and accuracy against known ground truth
6. **Export summaries** — format structured data into Markdown reports

## 3 — Problem Statement

**Problem:** After a 30-minute meeting, you need to send a summary email with decisions, action items, and blockers. Manual note-taking is error-prone — you miss details while participating.

**Constraints:**
- Transcripts are noisy (interruptions, tangents, filler words)
- Action items are scattered throughout, not listed at the end
- Deadlines and owners are often implicit ("I'll get that done by Thursday")
- A single naive summarization prompt loses detail on structured fields

**Our approach:**
```
Raw transcript -> Parse speakers -> Decomposed extraction:
    +-- Executive summary prompt
    +-- Decisions extraction prompt
    +-- Action items extraction prompt
    +-- Blockers extraction prompt
    +-- Open questions prompt
-> Merge into structured output -> Format as Markdown report
```

## 4 — Why This Project Matters

**Meeting summarization is a top-3 enterprise GenAI use case:**
- Otter.ai, Fireflies.ai, Microsoft Copilot, Google Gemini all offer this
- Saves 10-15 minutes per meeting x 5-10 meetings/week = hours per person
- Reduces "I thought you were handling that" miscommunication

**Prompt decomposition is an underrated production pattern:**

| Approach | Quality | Reliability | Debuggability |
|----------|---------|-------------|---------------|
| Single mega-prompt | Medium | Low | Hard — which part failed? |
| Decomposed sub-prompts | High | High — failures isolated | Easy — test each independently |

Most production LLM systems use decomposed prompts, not monolithic ones.

## 5 — Dataset Overview

We use **two manually crafted meeting transcripts** with known ground truth:

| Transcript | Type | Speakers | Key items |
|-----------|------|----------|-----------|
| Sprint Planning | Engineering standup | 6 people | 3 decisions, 5 action items, 3 blockers |
| Stakeholder Check-in | Executive update | 3 people | 2 decisions, 2 action items, 1 blocker |

Using hand-crafted transcripts is intentional: we know exactly what's present so we can evaluate extraction accuracy. No privacy concerns with real meeting data.

## 6 — Environment Setup

This notebook runs in **two modes**:
- **LLM mode**: Uses local Ollama for high-quality summarization (optional)
- **Fallback mode**: Deterministic rule-based extraction (always works)

No external API keys or paid services needed.

In [ ]:
import json
import re
import urllib.request
import urllib.error
from pathlib import Path
from typing import Any, Dict, List


def is_ollama_available(host: str = "http://localhost:11434") -> bool:
    """Check if Ollama server is running locally."""
    try:
        with urllib.request.urlopen(f"{host}/api/tags", timeout=2) as resp:
            return resp.status == 200
    except Exception:
        return False


OLLAMA_HOST = "http://localhost:11434"
OLLAMA_MODEL = "qwen3:8b"
USE_OLLAMA = is_ollama_available(OLLAMA_HOST)

print(f"Ollama available: {USE_OLLAMA}")
if not USE_OLLAMA:
    print("Running in fallback mode (rule-based extraction). All cells will execute.")


## 7 — Imports

All imports are standard library — already loaded in the setup cell above. No additional packages required.

## 8 — Data Loading (Sample Transcripts)

We define two transcripts with known ground-truth content so we can evaluate extraction quality later.

In [ ]:
TRANSCRIPT_1 = """
Sprint Planning Meeting - March 15, 2025

Participants: Sarah (PM), Mike (Tech Lead), Lisa (Frontend), James (Backend),
              Priya (QA), Tom (Designer)

Sarah: Good morning everyone. Let's review where we stand on the Q1 release.
Mike, can you give us the engineering update?

Mike: Sure. The API migration is about 70% done. We hit a snag with the
authentication service - the OAuth2 integration with the new provider is
taking longer than expected. James has been leading that effort.

James: Yeah, the main issue is that their sandbox environment has been
unreliable. I've been in contact with their support team. They promised a
fix by Wednesday. If that doesn't come through, we might need to consider
the fallback provider we evaluated last month.

Sarah: That's concerning. What's the impact on the timeline if we switch?

Mike: Roughly 5 business days of additional work. We'd push the release
from March 28 to April 4.

Sarah: OK, let's make a decision on this by Friday. James, keep pushing
with the current provider until Wednesday. If their fix doesn't land,
we'll go with the fallback. Mike, can you prepare the fallback
implementation plan just in case?

Mike: Will do. I'll have it ready by Thursday.

Sarah: Lisa, how's the frontend looking?

Lisa: The new dashboard is feature-complete. I'm waiting on Tom's final
designs for the settings page. Tom, any ETA on that?

Tom: I'll have the mockups done by end of day tomorrow. Sorry for the
delay - I was pulled into the marketing site redesign last week.

Lisa: No worries. Once I get the designs, I'll need about 3 days to
implement and test.

Sarah: Priya, how's the test plan?

Priya: I've written test cases for 80% of the new features. The main
gap is the OAuth flow - I need the integration working to write proper
E2E tests. I also found 3 critical bugs from last sprint that still
aren't fixed.

Mike: Which bugs?

Priya: The memory leak in the notification service, the race condition
in the payment queue, and the timezone display bug for European users.
The first two are P0s.

Sarah: James and Mike, can you prioritize those P0 bugs this sprint?
We can't ship with a memory leak.

Mike: Agreed. James, take the payment queue fix. I'll handle the
notification service memory leak.

James: Got it.

Sarah: Great. Let's plan to reconvene on Wednesday to check the OAuth
status. If anyone hits a blocker before then, ping me on Slack.

Tom: Quick question - are we still planning the user onboarding redesign
for Q2? I want to start research next week.

Sarah: Yes, that's still on the roadmap. Let's discuss it in next week's
planning session. Good point bringing it up.

Sarah: Alright, that's a wrap. Thanks everyone!
""".strip()


TRANSCRIPT_2 = """
Stakeholder Check-in - March 18, 2025

Participants: Anna (VP Product), Sarah (PM), Mike (Tech Lead)

Anna: Thanks for meeting on short notice. The board wants a demo of the
new dashboard at the April 10 all-hands. Is that feasible?

Sarah: The dashboard frontend is feature-complete. The risk is the OAuth
integration - we're still waiting on the third-party provider's fix.

Mike: If their fix lands by Wednesday as promised, we'll be on track for
March 28 release. If not, we switch to the fallback provider and push to
April 4. Either way, April 10 demo is doable.

Anna: Good. I want to make sure the demo shows the real-time analytics
feature. That's the one the board cares most about.

Mike: That part is solid. I'll make sure we have sample data loaded for
the demo environment.

Anna: One more thing - legal flagged that we need updated privacy consent
for the new user tracking. Sarah, can you coordinate with legal to get
the copy updated?

Sarah: Yes, I'll reach out to legal today. We should have that sorted
by end of next week.

Anna: Perfect. Let's do another check-in Friday to confirm the OAuth
decision. Thanks both.
""".strip()


# Ground truth for evaluation
GROUND_TRUTH_1 = {
    "num_decisions": 3,
    "num_action_items": 5,
    "num_blockers": 3,
    "attendees": ["Sarah", "Mike", "Lisa", "James", "Priya", "Tom"],
}


print(f"Transcript 1: {len(TRANSCRIPT_1.split())} words, {len(TRANSCRIPT_1)} chars")
print(f"Transcript 2: {len(TRANSCRIPT_2.split())} words, {len(TRANSCRIPT_2)} chars")


## 9 — Data Inspection

Let's analyze the transcript structure before building the summarizer.

In [ ]:
def inspect_transcript(text: str, label: str = "Transcript") -> Dict[str, Any]:
    """Analyze basic transcript structure."""
    lines = text.strip().split("\n")
    non_empty = [l for l in lines if l.strip()]

    speaker_pattern = re.compile(r"^(\w+):\s")
    speakers = set()
    turn_count = 0
    for line in non_empty:
        m = speaker_pattern.match(line)
        if m:
            speakers.add(m.group(1))
            turn_count += 1

    info = {
        "label": label,
        "total_lines": len(lines),
        "non_empty_lines": len(non_empty),
        "word_count": len(text.split()),
        "speakers": sorted(speakers),
        "speaker_turns": turn_count,
    }

    print(f"--- {label} ---")
    for k, v in info.items():
        if k != "label":
            print(f"  {k}: {v}")
    print()
    return info


info1 = inspect_transcript(TRANSCRIPT_1, "Sprint Planning")
info2 = inspect_transcript(TRANSCRIPT_2, "Stakeholder Check-in")


## 10 — Data Preprocessing

We extract speaker turns into a structured list. This makes downstream extraction cleaner for both LLM and rule-based approaches.

In [ ]:
def parse_speaker_turns(text: str) -> List[Dict[str, str]]:
    """Parse transcript into a list of {speaker, text} dicts."""
    speaker_pattern = re.compile(r"^(\w+):\s+(.*)")
    turns = []
    current_speaker = None
    current_text = []

    for line in text.strip().split("\n"):
        m = speaker_pattern.match(line.strip())
        if m:
            if current_speaker:
                turns.append({"speaker": current_speaker, "text": " ".join(current_text)})
            current_speaker = m.group(1)
            current_text = [m.group(2)]
        elif current_speaker and line.strip():
            current_text.append(line.strip())

    if current_speaker:
        turns.append({"speaker": current_speaker, "text": " ".join(current_text)})

    return turns


turns_1 = parse_speaker_turns(TRANSCRIPT_1)
print(f"Parsed {len(turns_1)} speaker turns from transcript 1")
print(f"\nFirst 3 turns:")
for t in turns_1[:3]:
    print(f"  {t['speaker']}: {t['text'][:80]}...")


## 11 — Baseline Approach (Rule-Based Extraction)

Before using an LLM, we build a **deterministic rule-based extractor**. This serves as:
- A **fallback** when Ollama is unavailable
- A **baseline** to compare LLM quality against
- A demonstration of **why LLMs add value** — rules miss implicit information (decisions, rationale, indirect assignments)

In [ ]:
def extract_attendees(text: str) -> List[str]:
    """Extract attendee names from the Participants line."""
    m = re.search(r"Participants?:\s*(.+?)(?:\n\n|\n[A-Z])", text, re.DOTALL)
    if m:
        raw = m.group(1).replace("\n", " ")
        names = re.findall(r"(\w+)\s*\(", raw)
        return names
    return []


def extract_action_items_rule(turns: List[Dict[str, str]]) -> List[Dict[str, str]]:
    """Extract action items using keyword heuristics."""
    action_keywords = [
        r"I'll\b", r"I will\b", r"will do\b", r"can you\b",
        r"take the\b", r"handle the\b", r"prepare the\b",
        r"reach out\b", r"prioritize\b",
    ]
    pattern = re.compile("|".join(action_keywords), re.IGNORECASE)

    items = []
    for turn in turns:
        if pattern.search(turn["text"]):
            deadline = "TBD"
            dl_match = re.search(
                r"by\s+(Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday"
                r"|end of day tomorrow|end of (?:next )?week|tomorrow|tonight"
                r"|March \d+|April \d+)",
                turn["text"], re.IGNORECASE,
            )
            if dl_match:
                deadline = dl_match.group(1)

            items.append({
                "owner": turn["speaker"],
                "task": turn["text"][:120],
                "deadline": deadline,
                "priority": "medium",
            })

    return items


def extract_blockers_rule(turns: List[Dict[str, str]]) -> List[str]:
    """Extract blockers using keyword heuristics."""
    blocker_kw = re.compile(
        r"snag|issue|unreliable|delay|block|wait|concern|risk|bug|critical|P0",
        re.IGNORECASE,
    )
    blockers = []
    for turn in turns:
        if blocker_kw.search(turn["text"]):
            snippet = turn["text"][:120]
            blockers.append(f"{turn['speaker']}: {snippet}")
    return blockers


def baseline_summarize(text: str) -> Dict[str, Any]:
    """Full rule-based extraction pipeline."""
    turns = parse_speaker_turns(text)
    attendees = extract_attendees(text)

    first_line = text.strip().split("\n")[0].strip()
    title = first_line if len(first_line) < 80 else "Meeting Summary"

    date_match = re.search(
        r"((?:January|February|March|April|May|June|July|August|September"
        r"|October|November|December)\s+\d{1,2},?\s+\d{4})", text
    )
    date = date_match.group(1) if date_match else "Not specified"

    return {
        "title": title,
        "date": date,
        "attendees": attendees,
        "executive_summary": f"Meeting with {len(attendees)} participants covering project status updates.",
        "decisions": [],
        "action_items": extract_action_items_rule(turns),
        "blockers": extract_blockers_rule(turns),
        "open_questions": [],
        "next_meeting": "Not extracted (rule-based)",
    }


baseline_result = baseline_summarize(TRANSCRIPT_1)
print("Baseline extraction results:")
print(f"  Attendees found: {baseline_result['attendees']}")
print(f"  Action items found: {len(baseline_result['action_items'])}")
print(f"  Blockers found: {len(baseline_result['blockers'])}")
print(f"  Decisions found: {len(baseline_result['decisions'])} (rules cannot extract rationale)")


## 12 — Main Model / Workflow (Decomposed Prompt Extraction)

### Why Decompose?

Instead of one massive prompt asking for everything, we use **separate focused prompts**:

| Sub-task | Why separate? |
|----------|--------------|
| Executive summary | Needs holistic view of entire meeting |
| Decisions | Must distinguish decisions from discussions |
| Action items | Must extract owner + task + deadline triples |
| Blockers | Must identify active risks vs. resolved issues |
| Open questions | Must find unanswered questions specifically |

Each sub-prompt is shorter, more focused, and easier to debug in isolation.

In [ ]:
def call_ollama_json(prompt: str) -> Any:
    """Call Ollama and parse the response as JSON."""
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "format": "json",
    }
    req = urllib.request.Request(
        f"{OLLAMA_HOST}/api/generate",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    raw = data.get("response", "{}")
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {}


print("LLM helper function defined.")


## 13 — Pipeline Execution (Decomposed Extraction)

We run each sub-prompt independently and merge results.

In [ ]:
PROMPT_EXEC_SUMMARY = """You are a professional meeting secretary. Read this transcript and write
a 2-3 sentence executive summary. Focus on the main topics discussed and
overall meeting outcome. Do not list individual action items.

Transcript:
{transcript}

Return JSON: {{"executive_summary": "..."}}"""

PROMPT_DECISIONS = """Extract all decisions made during this meeting. A decision is something
that was explicitly agreed on, not just discussed.

Transcript:
{transcript}

Return JSON: {{"decisions": [{{"decision": "...", "rationale": "..."}}]}}"""

PROMPT_ACTION_ITEMS = """Extract all action items from this meeting transcript. An action item must have:
- owner: who is responsible
- task: what they need to do
- deadline: when it is due (use "TBD" if not specified)
- priority: high, medium, or low

Transcript:
{transcript}

Return JSON: {{"action_items": [{{"owner": "...", "task": "...", "deadline": "...", "priority": "..."}}]}}"""

PROMPT_BLOCKERS = """Extract all blockers, risks, and issues mentioned in this meeting that
could delay or block progress. Be specific.

Transcript:
{transcript}

Return JSON: {{"blockers": ["..."]}}"""

PROMPT_OPEN_QUESTIONS = """Extract questions that were raised during this meeting but NOT fully
answered or resolved. Do NOT include rhetorical questions.

Transcript:
{transcript}

Return JSON: {{"open_questions": ["..."]}}"""


def llm_summarize(text: str) -> Dict[str, Any]:
    """Full LLM-powered decomposed extraction."""
    attendees = extract_attendees(text)
    first_line = text.strip().split("\n")[0].strip()
    title = first_line if len(first_line) < 80 else "Meeting Summary"
    date_match = re.search(
        r"((?:January|February|March|April|May|June|July|August|September"
        r"|October|November|December)\s+\d{1,2},?\s+\d{4})", text
    )
    date = date_match.group(1) if date_match else "Not specified"

    print("  Extracting executive summary...")
    exec_r = call_ollama_json(PROMPT_EXEC_SUMMARY.format(transcript=text))

    print("  Extracting decisions...")
    dec_r = call_ollama_json(PROMPT_DECISIONS.format(transcript=text))

    print("  Extracting action items...")
    act_r = call_ollama_json(PROMPT_ACTION_ITEMS.format(transcript=text))

    print("  Extracting blockers...")
    blk_r = call_ollama_json(PROMPT_BLOCKERS.format(transcript=text))

    print("  Extracting open questions...")
    oq_r = call_ollama_json(PROMPT_OPEN_QUESTIONS.format(transcript=text))

    return {
        "title": title,
        "date": date,
        "attendees": attendees,
        "executive_summary": exec_r.get("executive_summary", "Not available"),
        "decisions": dec_r.get("decisions", []),
        "action_items": act_r.get("action_items", []),
        "blockers": blk_r.get("blockers", []),
        "open_questions": oq_r.get("open_questions", []),
        "next_meeting": "See transcript for details",
    }


def summarize_meeting(text: str) -> Dict[str, Any]:
    """Main entry point: uses LLM if available, otherwise fallback."""
    if USE_OLLAMA:
        print("Using LLM-powered decomposed extraction...")
        return llm_summarize(text)
    else:
        print("Using rule-based fallback extraction...")
        return baseline_summarize(text)


print("Summarization pipeline ready.")


In [ ]:
# Run the full pipeline on transcript 1
print("=" * 60)
print("Processing: Sprint Planning Meeting")
print("=" * 60)
result_1 = summarize_meeting(TRANSCRIPT_1)
print("\nDone!")
print(json.dumps(result_1, indent=2, default=str))


## 14 — Evaluation and Interpretation

We evaluate:
1. **Output formatting** — display as a readable report
2. **Extraction quality** — compare extracted counts against known ground truth

In [ ]:
def display_summary(summary: Dict[str, Any]) -> None:
    """Display meeting summary in a readable format."""
    print("=" * 70)
    print(f"  {summary.get('title', 'Meeting Summary')}")
    print(f"  Date: {summary.get('date', 'N/A')}")
    print(f"  Attendees: {', '.join(summary.get('attendees', []))}")
    print("=" * 70)

    print(f"\n  EXECUTIVE SUMMARY")
    print(f"  {summary.get('executive_summary', 'N/A')}")

    decisions = summary.get("decisions", [])
    print(f"\n  DECISIONS ({len(decisions)})")
    for d in decisions:
        if isinstance(d, dict):
            print(f"    - {d.get('decision', '')}")
            rat = d.get("rationale", "")
            if rat:
                print(f"      Rationale: {rat}")
        else:
            print(f"    - {d}")

    actions = summary.get("action_items", [])
    print(f"\n  ACTION ITEMS ({len(actions)})")
    for a in actions:
        if isinstance(a, dict):
            pri = a.get("priority", "medium").upper()
            print(f"    [{pri}] {a.get('owner', '?')}: {a.get('task', '')[:100]}")
            print(f"           Due: {a.get('deadline', 'TBD')}")
        else:
            print(f"    - {a}")

    blockers = summary.get("blockers", [])
    print(f"\n  BLOCKERS ({len(blockers)})")
    for b in blockers:
        print(f"    ! {b}")

    questions = summary.get("open_questions", [])
    print(f"\n  OPEN QUESTIONS ({len(questions)})")
    for q in questions:
        print(f"    ? {q}")

    print(f"\n  NEXT MEETING: {summary.get('next_meeting', 'Not discussed')}")
    print("=" * 70)


display_summary(result_1)


In [ ]:
def evaluate_extraction(result: Dict[str, Any], ground_truth: Dict[str, Any], label: str) -> Dict[str, Any]:
    """Score extraction quality against known ground truth."""
    scores = {}

    extracted_att = set(result.get("attendees", []))
    expected_att = set(ground_truth.get("attendees", []))
    scores["attendee_recall"] = round(len(extracted_att & expected_att) / max(len(expected_att), 1), 2)

    n_actions = len(result.get("action_items", []))
    expected_actions = ground_truth.get("num_action_items", 0)
    scores["action_items_extracted"] = n_actions
    scores["action_items_expected"] = expected_actions
    scores["action_item_coverage"] = round(min(n_actions / max(expected_actions, 1), 1.0), 2)

    n_blockers = len(result.get("blockers", []))
    scores["blockers_extracted"] = n_blockers
    scores["blockers_expected"] = ground_truth.get("num_blockers", 0)

    n_decisions = len(result.get("decisions", []))
    scores["decisions_extracted"] = n_decisions
    scores["decisions_expected"] = ground_truth.get("num_decisions", 0)

    exec_sum = result.get("executive_summary", "")
    scores["has_exec_summary"] = bool(exec_sum and len(exec_sum) > 20)

    print(f"--- Evaluation: {label} ---")
    for k, v in scores.items():
        print(f"  {k}: {v}")
    print()
    return scores


eval_1 = evaluate_extraction(result_1, GROUND_TRUTH_1, "Sprint Planning")


In [ ]:
# Process transcript 2
print("=" * 60)
print("Processing: Stakeholder Check-in")
print("=" * 60)
result_2 = summarize_meeting(TRANSCRIPT_2)
display_summary(result_2)


### Export to Markdown

Convert the structured summary into a portable Markdown report file.

In [ ]:
def export_to_markdown(summary: Dict[str, Any], output_path: str = "meeting_summary.md") -> None:
    """Export summary to a Markdown file."""
    lines = [
        f"# {summary.get('title', 'Meeting Summary')}",
        "",
        f"**Date:** {summary.get('date', 'N/A')}  ",
        f"**Attendees:** {', '.join(summary.get('attendees', []))}",
        "",
        "## Executive Summary",
        summary.get("executive_summary", "N/A"),
        "",
    ]

    decisions = summary.get("decisions", [])
    if decisions:
        lines.append("## Decisions")
        for d in decisions:
            if isinstance(d, dict):
                lines.append(f"- **{d.get('decision', '')}** - {d.get('rationale', '')}")
            else:
                lines.append(f"- {d}")
        lines.append("")

    actions = summary.get("action_items", [])
    if actions:
        lines.append("## Action Items")
        lines.append("| Owner | Task | Deadline | Priority |")
        lines.append("|-------|------|----------|----------|")
        for a in actions:
            if isinstance(a, dict):
                lines.append(f"| {a.get('owner','')} | {a.get('task','')[:80]} | {a.get('deadline','TBD')} | {a.get('priority','medium')} |")
            else:
                lines.append(f"| - | {a} | TBD | medium |")
        lines.append("")

    blockers = summary.get("blockers", [])
    if blockers:
        lines.append("## Blockers")
        for b in blockers:
            lines.append(f"- {b}")
        lines.append("")

    questions = summary.get("open_questions", [])
    if questions:
        lines.append("## Open Questions")
        for q in questions:
            lines.append(f"- {q}")
        lines.append("")

    lines.append(f"**Next Meeting:** {summary.get('next_meeting', 'Not discussed')}")

    Path(output_path).write_text("\n".join(lines), encoding="utf-8")
    print(f"Exported to: {output_path}")


export_to_markdown(result_1, "meeting_summary.md")


## 15 — Error Analysis / Limitations

**Rule-based fallback limitations:**
- Cannot extract decisions (requires understanding context, not just keywords)
- Cannot infer implicit deadlines ("end of day tomorrow" requires date math)
- Cannot distinguish resolved issues from active blockers
- Misses action items phrased indirectly ("Can you handle that?" -> "Got it")

**LLM limitations:**
- May hallucinate action items not actually in the transcript
- May miss implicit assignments (agreement by silence)
- Quality varies by model size and temperature setting
- Long transcripts may exceed context window

**Evaluation limitations:**
- Ground truth counts are approximate (reasonable people may disagree on what counts as a "decision")
- We measure count completeness, not semantic accuracy of each extracted item

## 16 — How to Improve It

### Production Improvement Ideas

1. **Audio input pipeline**: Add Whisper speech-to-text before summarization
2. **Streaming extraction**: Process transcript in chunks for very long meetings (2+ hours)
3. **Action item tracking**: Store items in a database with status (open/done) and send reminders
4. **Follow-up detection**: Cross-reference with previous meeting summaries to flag overdue items
5. **Speaker diarization**: Auto-detect who is speaking from audio (no need for "Name:" format)
6. **Confidence scores**: Have the LLM rate its own confidence per extracted item
7. **Slack/Teams integration**: Auto-post summary to the meeting channel after the call ends

## Common Mistakes

1. **Single mega-prompt for everything**
   Asking the LLM to extract summary + decisions + actions + blockers + questions in one prompt reduces quality on each sub-task. Decompose instead.

2. **Not validating structured output**
   LLMs sometimes return malformed JSON or miss required fields. Always handle `json.JSONDecodeError` and provide defaults.

3. **Confusing discussions with decisions**
   "We should consider X" is not a decision. "Let's go with X" is. Prompt wording matters.

4. **Ignoring implicit action items**
   "Got it" after a request is an implicit acceptance. Rules miss this; LLMs sometimes do too.

5. **No fallback strategy**
   If the LLM is down, your pipeline breaks. Always have a deterministic fallback.

6. **Not testing with multiple transcript styles**
   A summarizer that works on structured meetings may fail on casual chat-style discussions.

## 17 — Practice Exercises

### Mini Challenge

**Challenge 1:** Write a new decomposed prompt that extracts **risks** separately from blockers. A risk is something that *might* happen; a blocker is something that *is* happening.

**Challenge 2:** Add a "sentiment" field to the summary — was the meeting positive, neutral, or tense? Design a prompt that classifies meeting tone.

**Challenge 3:** Take a real meeting transcript from your work (anonymized) and run it through the pipeline. Compare the output to your own manual notes. Where does the pipeline miss?

### Try This Next
- Add a follow-up email generator that uses the structured summary to draft a recap email
- Build a diff tool that compares two meeting summaries to track progress between sprints
- Add support for multi-language transcripts

## 18 — Final Takeaway / Summary

This notebook demonstrated:

1. **Prompt decomposition** — breaking one big extraction task into focused sub-prompts improves quality and debuggability
2. **Structured output** — using JSON schemas to get typed, validated data from LLMs
3. **Baseline comparison** — rule-based extraction shows why LLMs add value (decisions, context, implicit items)
4. **Graceful fallback** — the pipeline works end-to-end even without an LLM
5. **Evaluation without labels** — comparing extraction counts against known ground truth

**Key principle:** In production, always decompose complex extraction tasks into independent sub-prompts. It's more reliable, easier to debug, and lets you improve each component independently.